# Week 04

## Images

### Setup

Run the following 2 cells to import all necessary libraries and helpers for this week's exercises

In [ ]:
!wget -q "https://github.com/PSAM-5020-2026F-A/5020-utils/raw/main/src/image_utils.py" -O image_utils.py
!wget -q "https://github.com/PSAM-5020-2026F-A/WK04/raw/main/data.tgz" -O data.tgz
!tar -xzf data.tgz && rm -rf data.tgz

In [ ]:
import matplotlib.pyplot as plt
import random

from PIL import Image as PImage

from image_utils import blur, edges, get_pixels, make_image

## Images as lists (of lists) of pixels

Just a quick review of how images are usually represented and stored in files and memory.

An image:<br>
<img src="https://psam-5020-2026f-a.github.io/5020-imgs/WK04/pixel-00.jpg" height="250px">

is a collection of rows:<br>
<img src="https://psam-5020-2026f-a.github.io/5020-imgs/WK04/pixel-01.jpg" height="250px">

which are collections of pixels:<br>
<img src="https://psam-5020-2026f-a.github.io/5020-imgs/WK04/pixel-03.jpg" height="250px">

which are lists of color values:<br>
<img src="https://psam-5020-2026f-a.github.io/5020-imgs/WK04/pixel-04.jpg" height="250px">



### Filtering by Color

Let's formalize what we mean by filtering and be a bit more precise with what we are trying to do.

Let's say we're working on a vegetation detector and we want to be able to separate the pixels that represent plants and flowers from pixels that represent animals and other things.

We can start by creating a filter to separate the green pixels from our original image.

This is different than looking at the `green` color channel, or removing the `red` and `blue` channels, or exaggerating the green pixels.

In order to filter pixels of a certain color we have to go through the pixels and measure how similar they are to the color we wish to separate.

There are many ways to define "similar" when working with colors, but to keep it simple, let's define a `color_distance()` function that calculates the [Euclidean distance](https://en.wikipedia.org/wiki/Euclidean_distance) between two colors:

$\displaystyle dist = \sqrt{\left(R_0 - R_1\right)^2 + \left(G_0 - G_1\right)^2 + \left(B_0 - B_1\right)^2}$

In [ ]:
def color_distance(c0, c1):
  return ((c0[0] - c1[0])**2 + (c0[1] - c1[1])**2 + (c0[2] - c1[2])**2) ** 0.5

### Removing pixels

Now that we have a function for measuring color similarity we can go through the pixels and remove the ones that are very different from the color we want to keep. We'll remove pixels by turning them black with RGB value (0, 0, 0).

Since we're making some pretty significant changes to our image, let's keep a copy of the original. We can do this by just calling the `copy()` member function of an image object.

In [ ]:
himg = PImage.open("./data/hog.jpg")

keep_color = (20, 180, 20) # green
keep_color = (242, 224, 19) # yellow
thold = 120

filtpxs = []
for r,g,b in get_pixels(himg):
  if color_distance((r, g, b), keep_color) < thold:
    filtpxs.append((r, g, b))
  else:
    filtpxs.append((0, 0, 0))

fimg = make_image(filtpxs, fimg.size[0])

display(himg)
display(fimg)

### Filter any color

How can we filter the image to keep only the flowers? Or to keep only the hedgehog?

It might help to define a `filter_color()` function here that takes an image and a color to keep as inputs, and returns another image with just the kept pixels and black pixels ... while keeping the original image intact.

In [ ]:
def filter_color(img, keep_color, thold=150):
  # TODO: fill this in
  filtpxs = []

  return make_image(filtpxs, img.size[0])

In [ ]:
himg = PImage.open("./data/hog.jpg")

# TODO: use filter_color to filter image to keep only flowers or to keep only the hedgehog

## Features

### Counting pixels by colors

Now that we can separate pixels by color, we could use it to create an automatic deforestation sensor by separating and counting green pixels, and calculating the percentage of green areas on images.

We could implement a separate function to count the number of non-black pixels in an image after it has been filtered, but since our `filter_color()` function above already goes through all of the pixels in an image and detects pixels of specific colors, we can just create a modified version of it that counts those pixels and returns the ratio relative to the total number of pixels, instead of returning a filtered image.

We can call this new function `color_ratio()` and it will take and image, a color and a threshold as parameters, like the `filter_color()` function.

In [ ]:
# TODO: create color_ratio() function

def color_ratio(img, keep_color, thold=150):
  # TODO: modify the content of the filter_color() function
  return 0

We can try it out on a couple of forest images inside the `data/` directory.

In [ ]:
fimg = PImage.open("./data/forest-00.jpg")
display(fimg)

ffimg = filter_color(fimg, (0,200,0), 180)
display(ffimg)

green_ratio = color_ratio(fimg, (0,200,0), 180)
print(f"green %: {round(100 * green_ratio, 3)}")

### Try it out on some other images

There are $6$ other aerial forest images in the `data/` directory. Run the green pixel count code on them and see if the results make sense.

In [ ]:
# TODO: count green pixels in forest images
fimg = PImage.open("./data/forest-01.jpg")
display(fimg)

# TODO: filter by color to get good RGB values

# TODO: use RGB values to calculate color ratio

Once we start doing image analysis it's good to be able to extract different kinds of information from our images in case we want to categorize them, filter them further or retrieve them from large databases later.

This is kind of equivalent to how we extracted amplitude and frequency features from audio files.

### Dominant Channel

One feature we can easily extract from our images is the average value of each of its channels along with the average luminosity value.

This can be used to give us some idea about the dominant color or tones in an images.

In [ ]:
himg = PImage.open("./data/hog.jpg")
himg_pxs = get_pixels(himg)

# array with 4 0s
hog_rgbl_sum = 4 * [0]

for r,g,b in himg_pxs:
  l = (r + g + b) // 3
  hog_rgbl_sum[0] += r
  hog_rgbl_sum[1] += g
  hog_rgbl_sum[2] += b
  hog_rgbl_sum[3] += l

hog_rgbl_avg = [s // len(himg_pxs) for s in hog_rgbl_sum]

print("average RGBL values", hog_rgbl_avg)

We can see that both the `green` and `red` channels have average values above the average luminosity value.

This makes sense since the image has a lot of green pixels, and the `red` channel contributes to the yellow and white pixels.

### Repeat for other image

Get the average value for each channel of a different image.

Maybe create a function...

Does the result make sense?

In [ ]:
# TODO: get average channel value for other image

def get_channel_avgs(img):
  # TODO: fill this in
  return []

In [ ]:
# TODO: run function on image and print channel average values

In [ ]:
# TODO: run function on another image and print channel average values

## Bonus Feature

### Edge Detection

We've looked at some techniques for getting color information from images, but images are more than just colors.

We might be interested in also quantifying the shapes and textures present in our images.

We can start by extracting the edges of shapes in our image. There are many ways of doing this, but the simplest way is to subtract our original image from a blurry version of it and threshold the result.

Since we are not so concerned with color at this point we should work with grayscale images.

Our overall algorithm will be something like:
- open an image
- make it black & white
- blur it
- subtract the blurry b&w pixels from the original b&w pixels
- threshold the result

Threshold means making slightly bright pixels really bright and all other pixels really dark.

Let's do this in steps:

#### Open an image and extract its pixels

We can use the `PImage.open()` and the `get_pixels()` functions to open and extract pixels.

In [ ]:
# TODO: implement edge extraction algorithm
# TODO: open an image and extract its pixels

mimg = '''image goes here'''
ipxs = '''pixels goes here'''

display(mimg)

#### Blur the image using `blur()`

And display it.

The `blur()` function takes an image object as a parameter and an optional second parameter that specifies the amount of blurring. It returns another image object.

Experiment with the parameter a little bit, but the default value is good for extracting edges.

Let's also get the pixels for the blurred image and display it with `display()`.

In [ ]:
# TODO: implement edge extraction algorithm
# TODO: blur image with the blur() function

bimg = ''''''
bpxs = ''''''

display(bimg)

#### Make the images grayscale

We saw this a few cells back. We can average the `RGB` values to get a grey luminance value.

Get grayscale versions of the original image and the blurry image

Display the results.

In [ ]:
# TODO: implement edge extraction algorithm
# TODO: make the image b&w

# TODO: make pixels and blurry pixels b&w
bwpxs = ''''''
bwbpxs = ''''''

bwimg = make_image(bwpxs, mimg.size[0])
display(bwimg)

bwbimg = make_image(bwbpxs, bimg.size[0])
display(bwbimg)

#### Subtract the blurred pixels from the original pixel values

The `zip()` function might help iterate through the pixel arrays from both images at the same time.

Display the resulting image.

In [ ]:
# TODO: implement edge extraction algorithm
# TODO: subtract blurry b&w image from original b&w image

# TODO: subtract bwbpxs from bwpxs
spxs = ''''''

simg = make_image(spxs, bwimg.size[0])
display(simg)

#### Threshold the resulting pixel values

We'll go through the array and check each pixel:<br>
if its luminance is greater than a threshold value, we'll make it $255$, otherwise we'll make it $0$.

We can start with a threshold value of $16$ and see what happens.

In [ ]:
# TODO: implement edge extraction algorithm
# TODO: threshold pixel values

# TODO: threshold pixels
tpxs = ''''''

timg = make_image(tpxs, simg.size[0])
display(timg)

Great !

### Let's repeat it for a different image

First, create a function that takes an image as a parameter and returns another image with edge information.

We just have to wrap up the previous steps into a function. It will be shorter since we don't have to display all the images along the way. That was only while debugging the logic.

In [ ]:
# TODO: create edge extraction function

def edge(img, rad=1, thold=16):
  # TODO: fill this in
  bimg = '''blurry image'''
  bwpxs = '''b&w pixels'''
  bbwpxs = '''blurry b&w pixels'''

  dpxs = '''difference between b&w and blurry-b&w pixels'''
  tpxs = '''threshold-ed difference'''

  return make_image(tpxs, img.size[0])

In [ ]:
# TODO: test function on a couple of images using different parameters

### Count edges

It helps to have a single value that we can use to compare edge information between images.

Let's create a function that counts the number of white pixels in an edge-extraction image.

We'll divide this number by the number of pixels in the image to get a rough idea of how _edgy_ any image is.

In [ ]:
def edge_ratio(img, rad=1, thold=16):
  eimg = edge(img, rad=rad, thold=thold)
  eimg_pxs = get_pixels(eimg)
  sum255 = sum([1 for L in eimg_pxs if L > 250])
  npxs = len(eimg_pxs)
  return sum255 / npxs

mimg = PImage.open("./data/hog.jpg")
display(edge(mimg, 2))
round(100*edge_ratio(mimg, 2), 4)

### Count edges for different images

Do the results make sense? Why did we divide the sum by the total number of pixels?

In [ ]:
# TODO: run the edge_ratio() function on a few images and compare results.
# TODO: do the results make sense ? (i.e. do images with more "things" have more edges ?)

## Bonus Filter

### Dithering

This is a visual filter that reduces the number of colors used to represent our image.

For grayscale images, we're only going to use the numbers $0$ and $255$ for pixel values. Regular grayscale images can use any value between $0$ and $255$, but with dithering we'll reduce that to just $0$ and $255$.

The simplest way to do this would be to threshold each pixel individually: if it's luminosity is greater than some threshold, make it white, otherwise, make it black.

In [ ]:
img = PImage.open("./data/hog.jpg")

dpxs = []
thold = 128

for r,g,b in get_pixels(img):
  l = (r + g + b) // 3
  if l < thold:
    dpxs.append(0)
  else:
    dpxs.append(255)

display(make_image(dpxs, img.width))

This is actually called thresholding, (or binning) and can be very useful in certain shape detection algorithms.

It really doesn't work well when the colors transition between light and dark. We want to improve this so we can keep more information from our image.

#### Dithering Maps

Instead of looking at a single pixel, we'll look at groups of neighboring pixels and threshold them together using slightly different threshold values for each pixel. The idea behind this is that if we're looking at a region that has $4$ dark gray pixels, we want our result to be $3$ black pixels and $1$ white pixel.

We achieve this using pre-computed threshold maps. These maps will be the same size as our image and we'll just go down the pixel list and the map list and check whether the pixel value is greater or smaller than the threshold value in the map list.

[Wikipedia](https://en.wikipedia.org/wiki/Ordered_dithering#Threshold_map) has some examples of maps of different sizes. We'll start with the $2 \times 2$ map. The idea is that we create a pixel list that has this pattern repeated for as many times as we need to cover our image rows and columns:

$\begin{bmatrix}
    0 & 128 \\
    192 & 64
\end{bmatrix}$

For our hedgehog image, with $1000$ by $600$ pixels, we'd have to repeat the pattern $500$ times horizontally and $300$ times vertically.

Since our pixel are in one-dimensional lists, what this means is that we repeat $[0,128]$ $500$ times, followed by $[192,64]$ $500$ times:

$[0, 128, 0, 128, 0, 128, ... (\text{repeat 500 times}) ... 192, 64, 192, 64, 192, 64, ... (\text{repeat 500 times}) ... ]$

These $2000$ numbers represent $2$ rows worth of pixels. To fully cover our image we repeat what we have so far $300$ times, to cover the $600$ rows.

Good thing we can do list algebra in `Python` !

In [ ]:
dit_map_2_rows = (img.width//2) * [0, 128] + (img.width//2) * [192, 64]
print(len(dit_map_2_rows))

dit_map_2x2 = (img.height // 2) * dit_map_2_rows
print(len(dit_map_2x2))

#### Threshold with maps

Using `zip` to iterate through both lists at the same time.

In [ ]:
img = PImage.open("./data/hog.jpg")
pxs = get_pixels(img)

dpxs = []

for (r,g,b),thold in zip(pxs, dit_map_2x2):
  l = (r + g + b) // 3
  if l < thold:
    dpxs.append(0)
  else:
    dpxs.append(255)

display(make_image(dpxs, img.width))

#### More Maps !

All the pixels in the image above are black or white. We just perceive them as gray due to the density of white/black pixels in a given region.

So cool !

We can try out some of the other maps from [Wikipedia](https://en.wikipedia.org/wiki/Ordered_dithering#Threshold_map). The numbers there are for pixel values between $0$ and $1$ instead of $0$ and $255$, but we can get them in the range we need by multiplying them by $256$. So, for the $4 \times 4$ example, instead of multiplying by $\frac{1}{16}$, we would multiply by $\frac{256}{16}$, or $16$:

$\begin{bmatrix}
    0 & 128 & 32 & 160 \\
    192 & 64 & 224 & 96 \\
    48 & 176 & 16 & 144 \\
    240 & 112 & 208 & 80 \\
\end{bmatrix}$

In [ ]:
dit_map_4_rows = (
  (img.width//4) * [0, 128, 32, 160] +
  (img.width//4) * [192, 64, 224, 96] +
  (img.width//4) * [48, 176, 16, 144] +
  (img.width//4) * [240, 112, 208, 80]
)
print(len(dit_map_4_rows))

dit_map_4x4 = (img.height // 4) * dit_map_4_rows
print(len(dit_map_4x4))

In [ ]:
img = PImage.open("./data/hog.jpg")
pxs = get_pixels(img)

dpxs = []

for (r,g,b),thold in zip(pxs, dit_map_4x4):
  l = (r + g + b) // 3
  # we can replace the if/else with a ternary operator
  dpxs.append(0 if l < thold else 255)

display(make_image(dpxs, img.width))

#### Other Strategies

The $4 \times 4$ matrix we constructed above is known as _Bayer Dithering_, and it preserves certain qualities of the original image (like overall luminosity), but there are other ways of representing images using only $2$ possible values per pixel that prioritize other properties of the image or the result.

Clustered-Dot (or halftone) dithering is the classic newspaper dithering strategy that prioritizes the distance between different black regions, in order to avoid ink smudges.

Void-and-Cluster dithering is another approach that uses a non-symmetrical matrix to achieve more smooth transitions between light and dark regions of the image.

In [ ]:
# TODO: Experiment here

dit_map_4_rows = (
  (img.width//4) * [0, 128, 32, 160] +
  (img.width//4) * [48, 176, 64, 144] +
  (img.width//4) * [192, 16, 224, 96] +
  (img.width//4) * [240, 112, 208, 80]
)
print(len(dit_map_4_rows))

dit_map_4x4 = (img.height // 4) * dit_map_4_rows
print(len(dit_map_4x4))

img = PImage.open("./data/hog.jpg")
pxs = get_pixels(img)

dpxs = []

for (r,g,b),thold in zip(pxs, dit_map_4x4):
  l = (r + g + b) // 3
  dpxs.append(0 if l < thold else 255)

display(make_image(dpxs, img.width))